# Gaussian Mixture Models (GMM) from Scratch ☁️

In this notebook, we implement **Gaussian Mixture Models (GMM)**, a probabilistic, soft clustering approach using the **Expectation-Maximization (EM)** algorithm.

## 📖 Theoretical Background

Unlike K-Means (hard clustering), GMM assumes data is generated from a mixture of $K$ Gaussian distributions, providing a probability of each point belonging to each cluster.

### 1. The EM Algorithm
**Initialization**: Randomly initialize means $\mu_k$, covariances $\Sigma_k$, and mixing weights $\pi_k$.

**E-Step (Expectation)**: Calculate the "responsibility" $\gamma_{ik}$, the probability that point $x_i$ belongs to cluster $k$:
$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_i | \mu_j, \Sigma_j)}$$

**M-Step (Maximization)**: Update the parameters using the responsibilities:
- $N_k = \sum_{i=1}^{N} \gamma_{ik}$ (Effective number of points in cluster $k$)
- $\mu_k = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} x_i$
- $\Sigma_k = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T$
- $\pi_k = \frac{N_k}{N}$

Repeat until convergence (Log-Likelihood stops increasing).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

np.random.seed(42)

## 🏗️ The Implementation

In [ ]:
class GMM:
    def __init__(self, k=3, max_iters=100, tol=1e-4):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        
    def _multivariate_gaussian(self, X, mu, cov):
        n = X.shape[1]
        diff = X - mu
        # Add small value to diagonal to prevent singular matrix
        cov += np.eye(n) * 1e-6
        inv_cov = np.linalg.inv(cov)
        det_cov = np.linalg.det(cov)
        
        norm_const = 1.0 / (np.power((2 * np.pi), n / 2.0) * np.sqrt(det_cov))
        result = np.exp(-0.5 * np.sum(np.dot(diff, inv_cov) * diff, axis=1))
        return norm_const * result
        
    def fit(self, X):
        self.n_samples, self.n_features = X.shape
        
        # 1. Initialization
        # Randomly choose initial means from data points
        random_idx = np.random.choice(self.n_samples, self.k, replace=False)
        self.means = X[random_idx]
        
        # Initialize covariances as identity matrices
        self.covs = [np.eye(self.n_features) for _ in range(self.k)]
        
        # Initialize mixing weights uniformly
        self.pis = np.ones(self.k) / self.k
        
        self.log_likelihoods = []
        
        for i in range(self.max_iters):
            # --- E-Step ---
            responsibilities = np.zeros((self.n_samples, self.k))
            
            for c in range(self.k):
                responsibilities[:, c] = self.pis[c] * self._multivariate_gaussian(X, self.means[c], self.covs[c])
                
            # Calculate log likelihood to check convergence
            log_likelihood = np.sum(np.log(np.sum(responsibilities, axis=1)))
            self.log_likelihoods.append(log_likelihood)
            
            # Normalize responsibilities (gamma)
            responsibilities = responsibilities / np.sum(responsibilities, axis=1, keepdims=True)
            
            # --- M-Step ---
            N_k = np.sum(responsibilities, axis=0)
            
            for c in range(self.k):
                # Update means
                self.means[c] = np.sum(responsibilities[:, c:c+1] * X, axis=0) / N_k[c]
                
                # Update covariances
                diff = X - self.means[c]
                self.covs[c] = np.dot((responsibilities[:, c:c+1] * diff).T, diff) / N_k[c]
                
                # Update pis
                self.pis[c] = N_k[c] / self.n_samples
                
            # Check convergence
            if i > 0 and np.abs(self.log_likelihoods[-1] - self.log_likelihoods[-2]) < self.tol:
                break

    def predict_proba(self, X):
        responsibilities = np.zeros((X.shape[0], self.k))
        for c in range(self.k):
            responsibilities[:, c] = self.pis[c] * self._multivariate_gaussian(X, self.means[c], self.covs[c])
        responsibilities = responsibilities / np.sum(responsibilities, axis=1, keepdims=True)
        return responsibilities
        
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

## 🧪 Data Generation and Training

In [ ]:
X, y = make_blobs(n_samples=400, centers=3, cluster_std=1.5, random_state=42)

# Stretch one of the clusters to make it elliptical
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_stretched = np.dot(X, transformation)

gmm = GMM(k=3, max_iters=100)
gmm.fit(X_stretched)
predictions = gmm.predict(X_stretched)

## 📊 Visualization

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(gmm.log_likelihoods)
plt.title("Log-Likelihood History")
plt.xlabel("Iteration")
plt.ylabel("Log-Likelihood")

plt.subplot(1, 2, 2)
plt.scatter(X_stretched[:, 0], X_stretched[:, 1], c=predictions, cmap="viridis", edgecolors='k', alpha=0.6)
plt.scatter(gmm.means[:, 0], gmm.means[:, 1], c='red', marker='X', s=200, label='Means')
plt.title("GMM Clustering Result")
plt.legend()
plt.show()